In [2]:
import nibabel
from tqdm import tqdm
import pydicom
import os
import matplotlib.pyplot as plt
import plotly.express as px

In [3]:
def convertNsave(arr,file_dir,fileName, index=0):
    """
    `arr`: parameter will take a numpy array that represents only one slice.
    `file_dir`: parameter will take the path to save the slices
    `index`: parameter will represent the index of the slice, so this parameter will be used to put 
    the name of each slice while using a for loop to convert all the slices
    """
    
    dicom_file = pydicom.dcmread('images/dcmimage.dcm')
    arr = arr.astype('uint16')
    dicom_file.Rows = arr.shape[0]
    dicom_file.Columns = arr.shape[1]
    dicom_file.PhotometricInterpretation = "MONOCHROME2"
    dicom_file.SamplesPerPixel = 1
    dicom_file.BitsStored = 16
    dicom_file.BitsAllocated = 16
    dicom_file.HighBit = 15
    dicom_file.PixelRepresentation = 1
    dicom_file.PixelData = arr.tobytes()
    dicom_file.save_as(os.path.join(file_dir, f'{fileName}slice{index}.dcm'))

def nifti2dicom_1file(nifti_dir, out_dir, fileName, dukeNamesDf):
    # nifti_dir is the directory, ex ./DataNiiFormat
    nifti_dir = nifti_dir + '/' + fileName + '.nii'
    nifti_file = nibabel.load(nifti_dir)
    nifti_array = nifti_file.get_fdata()
    number_slices = nifti_array.shape[2]

    for slice_ in tqdm(range(number_slices)):
        if int(dukeNamesDf[dukeNamesDf['Patient Name']==fileName]['Start Slice']) <= slice_ <= int(dukeNamesDf[dukeNamesDf['Patient Name']==fileName]['End Slice'])
        convertNsave(nifti_array[:,:,slice_], out_dir, fileName, slice_)

In [4]:
nifti2dicom_1file('./DataNiiFormat', './ConvertedNiftiToDicom', 'pat002')

100%|██████████| 142/142 [00:00<00:00, 300.79it/s]


In [5]:
file_path = './ConvertedNiftiToDicom/pat002slice65.dcm'
dicom_file = pydicom.read_file(file_path)
pixel_data = dicom_file.pixel_array
fig = px.imshow(pixel_data , color_continuous_scale = 'gray')
fig.update_layout(title="DICOM Image", xaxis_title="X", yaxis_title="Y")
fig.show()